# OPM Call/Price Range Optimizer

Reusable end-to-end notebook: given a spreadsheet with P/N, Total Calls, and DN Price columns, it:

1. Searches for **Call Range** breakpoints (A/B/C/D, given LA=2-3, C1=1, C0=0 are fixed) so that, across the full dataset:
   - Count of P/N:  D < C < B < A < LA < C1 < C0   (increasing)
   - Total Calls:   D > C > B > A > LA > C1 > C0   (decreasing)
2. Searches for **DN Price** breakpoints (4 limits -> 5 bands: to L1, to L2, to L3, to L4, high value), trying to keep L1 at or as close as possible (from above, if infeasible) to a target cap, so that:
   - Count of P/N:  band1 > band2 > band3 > band4 > band5  (decreasing)
   - Total Calls:   band1 > band2 > band3 > band4 > band5  (decreasing)
3. Builds a new workbook with the raw data, the breakpoints as editable blue inputs, and two live-formula pivot tables (Count of P/N, Total Calls) cross-tabbed by Price Range (rows) x Call Range (columns).

Run the cells in order, then edit the **Configuration** cell with your file paths/column names and run the **Execute** cell at the bottom.

In [ ]:
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT = "Arial"
CALL_CATS = ["D", "C", "B", "A", "LA", "C1", "C0"]

## Data loading

In [ ]:
def load_data(path, sheet=None, pn_col=None, price_col=None, calls_col=None):
    xls = pd.ExcelFile(path)
    sheet = sheet or xls.sheet_names[0]
    raw = pd.read_excel(path, sheet_name=sheet)

    def find_col(explicit, keyword):
        if explicit:
            return explicit
        for c in raw.columns:
            if keyword.lower() in str(c).lower():
                return c
        raise ValueError(f"Could not auto-detect a column matching '{keyword}'. "
                          f"Available columns: {list(raw.columns)}")

    pn_col = find_col(pn_col, "P/N")
    price_col = find_col(price_col, "Price")
    calls_col = find_col(calls_col, "Call")

    df = raw[[pn_col, price_col, calls_col]].copy()
    df.columns = ["PN", "Price", "Calls"]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)
    df["Calls"] = pd.to_numeric(df["Calls"], errors="coerce").fillna(0).astype(int)
    return df

## Call Range breakpoint search (A/B/C/D boundaries, given LA/C1/C0 fixed)

In [ ]:
def find_call_breakpoints(df):
    LA_count = int(df["Calls"].isin([2, 3]).sum())
    LA_sum = int(df.loc[df["Calls"].isin([2, 3]), "Calls"].sum())

    sub = df[df["Calls"] >= 4]
    if sub.empty:
        raise ValueError("No items with Calls >= 4; cannot build A/B/C/D bands.")
    calls_arr = sub["Calls"].values
    vals = np.sort(sub["Calls"].unique()).tolist()

    def band_stats(lo, hi):
        mask = (calls_arr >= lo) & (calls_arr <= hi)
        return int(mask.sum()), int(calls_arr[mask].sum())

    results = []
    for ai, a_max in enumerate(vals):
        cntA, sumA = band_stats(4, a_max)
        if cntA >= LA_count or sumA <= LA_sum:
            continue
        for bi in range(ai + 1, len(vals)):
            b_max = vals[bi]
            cntB, sumB = band_stats(a_max + 1, b_max)
            if cntB >= cntA or sumB <= sumA:
                continue
            for ci in range(bi + 1, len(vals)):
                c_max = vals[ci]
                cntC, sumC = band_stats(b_max + 1, c_max)
                if cntC >= cntB or sumC <= sumB:
                    continue
                cntD, sumD = band_stats(c_max + 1, vals[-1])
                if cntD == 0 or cntD >= cntC or sumD <= sumC:
                    continue
                minratio = min(cntB / cntA, cntC / cntB, cntD / cntC, cntA / LA_count)
                results.append((minratio, a_max, b_max, c_max))

    if not results:
        raise ValueError("No valid Call Range breakpoints found for this dataset "
                          "(the A/B/C/D/LA/C1/C0 monotonic pattern isn't achievable).")
    results.sort(key=lambda x: x[0])
    _, a_max, b_max, c_max = results[0]
    return a_max, b_max, c_max

## Price Range breakpoint search (4 limits -> 5 bands)

In [ ]:
def find_price_breakpoints(df, price_cap=10, whole_number=False, max_search=None, force_cap=True):
    """
    Searches for the 4 price breakpoints (Limit of range 1-4).

    Limit of range 1 is capped at `price_cap`. The search first tries to find
    a value at or below the cap where the full pattern holds (band1 has more
    items AND more total calls than every other band, and bands 2-5 are each
    strictly smaller than the one before). If no such value exists:
      - force_cap=True (default): Limit of range 1 is forced to `price_cap`
        anyway, and Limits 2-4 are chosen to keep bands 2-5 as sensibly
        decreasing as possible among themselves. A warning is printed
        explaining the pattern isn't fully satisfied.
      - force_cap=False: raises a ValueError instead.
    """
    pdata = df[df["Price"] > 0].copy()
    if pdata.empty:
        raise ValueError("No items with Price > 0; cannot build price bands.")

    if whole_number:
        # Round to whole numbers (minimum 1) so all breakpoints come out as integers
        pdata["Price"] = np.maximum(1, np.round(pdata["Price"].values)).astype(int)

    vals, inv = np.unique(pdata["Price"].values, return_inverse=True)
    cnt_per_val = np.bincount(inv, minlength=len(vals))
    sum_per_val = np.bincount(inv, weights=pdata["Calls"].values, minlength=len(vals))
    cum_cnt = np.cumsum(cnt_per_val)
    cum_sum = np.cumsum(sum_per_val)
    N = len(vals)
    total_cnt = int(cum_cnt[-1])
    total_sum = float(cum_sum[-1])

    def cum_at(i):
        if i < 0:
            return 0, 0.0
        return int(cum_cnt[i]), float(cum_sum[i])

    def band(prev_idx, idx):
        pc, ps = cum_at(prev_idx)
        c, s = cum_at(idx)
        return c - pc, s - ps

    def greedy_max(prev_idx, prev_cnt, prev_sum, min_start, max_idx):
        lo, hi, best = min_start, max_idx, None
        while lo <= hi:
            mid = (lo + hi) // 2
            bc, bs = band(prev_idx, mid)
            if bc < prev_cnt and bs < prev_sum and bc > 0:
                best = mid
                lo = mid + 1
            else:
                hi = mid - 1
        return best

    def dfs_from(idx1, tries_budget=150, ignore_prev=False):
        if ignore_prev:
            # Treat band1 as unconstrained: bands 2-5 just need to be
            # strictly decreasing among themselves.
            cnt1, sum1 = total_cnt + 1, total_sum + 1.0
        else:
            cnt1, sum1 = cum_at(idx1)
            if cnt1 == 0:
                return None
        max2 = greedy_max(idx1, cnt1, sum1, idx1 + 1, N - 2)
        if max2 is None:
            return None
        idx2, t2 = max2, 0
        while idx2 > idx1 and t2 < tries_budget:
            cnt2, sum2 = band(idx1, idx2)
            if 0 < cnt2 < cnt1 and sum2 < sum1:
                max3 = greedy_max(idx2, cnt2, sum2, idx2 + 1, N - 2)
                if max3 is not None:
                    idx3, t3 = max3, 0
                    while idx3 > idx2 and t3 < tries_budget:
                        cnt3, sum3 = band(idx2, idx3)
                        if 0 < cnt3 < cnt2 and sum3 < sum2:
                            max4 = greedy_max(idx3, cnt3, sum3, idx3 + 1, N - 1)
                            if max4 is not None:
                                idx4, t4 = max4, 0
                                while idx4 > idx3 and t4 < tries_budget:
                                    cnt4, sum4 = band(idx3, idx4)
                                    if 0 < cnt4 < cnt3 and sum4 < sum3:
                                        cnt5 = total_cnt - cum_at(idx4)[0]
                                        sum5 = total_sum - cum_at(idx4)[1]
                                        if 0 < cnt5 < cnt4 and sum5 < sum4:
                                            return (idx1, idx2, idx3, idx4)
                                    idx4 -= 1
                                    t4 += 1
                        idx3 -= 1
                        t3 += 1
            idx2 -= 1
            t2 += 1
        return None

    # Limit of range 1 must be <= price_cap: only consider values at or below
    # the cap, trying the ones closest to the cap first and working downward.
    cap_idx_candidates = np.where(vals <= price_cap)[0]
    if len(cap_idx_candidates) == 0:
        raise ValueError(
            f"No price values <= price_cap ({price_cap}); cannot build "
            f"a Limit of range 1 at or below the cap."
        )
    candidates = sorted(cap_idx_candidates.tolist(), reverse=True)
    candidates_search = candidates[:max_search] if max_search is not None else candidates

    # === Strict search: full pattern satisfied, Limit 1 <= price_cap ===
    sol = None
    for idx1 in candidates_search:
        if idx1 >= N - 3:
            continue
        sol = dfs_from(idx1)
        if sol:
            break

    if sol:
        i1, i2, i3, i4 = sol
        if whole_number:
            return int(vals[i1]), int(vals[i2]), int(vals[i3]), int(vals[i4])
        return float(vals[i1]), float(vals[i2]), float(vals[i3]), float(vals[i4])

    if not force_cap:
        raise ValueError(f"No valid Price Range breakpoints found with Limit of range 1 <= {price_cap}.")

    # === Strict search failed: force Limit 1 = price_cap anyway ===
    print(f"HEADS UP: no split satisfies the full decreasing pattern with Limit of range 1 <= {price_cap}.")
    print(f"Forcing Limit of range 1 = {price_cap} anyway; Limits 2-4 are best-effort and the "
          f"full band1-is-largest pattern is NOT guaranteed to hold.")

    idx1 = candidates[0]  # closest actual price value at or below the cap
    sol = dfs_from(idx1, ignore_prev=True)
    if sol:
        _, i2, i3, i4 = sol
        if whole_number:
            return int(round(price_cap)), int(vals[i2]), int(vals[i3]), int(vals[i4])
        return float(price_cap), float(vals[i2]), float(vals[i3]), float(vals[i4])

    # Last resort: not even bands 2-5 can be made to decrease among
    # themselves. Fall back to a simple decreasing-size split by position.
    print("Even bands 2-5 alone couldn't be made strictly decreasing - using an "
          "approximate positional split instead.")
    remaining_n = N - 1 - idx1
    if remaining_n < 4:
        raise ValueError("Not enough distinct price values above the cap to build the remaining bands.")
    fracs = [0.40, 0.68, 0.87]
    i2, i3, i4 = [idx1 + max(1, int(round(f * remaining_n))) for f in fracs]
    i2 = min(max(i2, idx1 + 1), N - 3)
    i3 = min(max(i3, i2 + 1), N - 2)
    i4 = min(max(i4, i3 + 1), N - 1)
    if whole_number:
        return int(round(price_cap)), int(vals[i2]), int(vals[i3]), int(vals[i4])
    return float(price_cap), float(vals[i2]), float(vals[i3]), float(vals[i4])

## Workbook builder

In [ ]:
def build_workbook(df, call_bps, price_bps, output_path, whole_number_price=False):
    a_max, b_max, c_max = call_bps
    p1, p2, p3, p4 = price_bps
    n = len(df)
    last_row = 1 + n

    wb = Workbook()
    ws = wb.active
    ws.title = "OPM_Optimized"

    blue = Font(name=FONT, color="0000FF")
    black = Font(name=FONT, color="000000")
    bold = Font(name=FONT, bold=True)
    header_fill = PatternFill("solid", start_color="D9E1F2")
    title_font = Font(name=FONT, bold=True, size=12)
    thin = Side(style="thin", color="BFBFBF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    headers = ["P/N", "Tot Call", "DN Price", "Call Range", "Price Range"]
    for i, h in enumerate(headers, start=1):
        c = ws.cell(row=1, column=i, value=h)
        c.font = bold
        c.fill = header_fill

    for idx, row in enumerate(df.itertuples(index=False), start=2):
        ws.cell(row=idx, column=1, value=row.PN).font = black
        ws.cell(row=idx, column=2, value=int(row.Calls)).font = black
        ws.cell(row=idx, column=2).number_format = "#,##0"
        ws.cell(row=idx, column=3, value=float(row.Price)).font = black
        ws.cell(row=idx, column=3).number_format = "#,##0.00"
        ws.cell(row=idx, column=4,
                value=f'=IF(B{idx}=0,"C0",IF(B{idx}=1,"C1",IF(B{idx}<=3,"LA",'
                      f'IF(B{idx}<=$H$2,"A",IF(B{idx}<=$H$3,"B",IF(B{idx}<=$H$4,"C","D"))))))').font = black
        ws.cell(row=idx, column=5,
                value=f'=IF(OR(C{idx}="",C{idx}<=0),"",IF(C{idx}<=$K$2,"to "&$K$2,'
                      f'IF(C{idx}<=$K$3,"to "&$K$3,IF(C{idx}<=$K$4,"to "&$K$4,'
                      f'IF(C{idx}<=$K$5,"to "&$K$5,"high value")))))').font = black

    for col, w in zip("ABCDE", (16, 10, 11, 12, 12)):
        ws.column_dimensions[col].width = w

    ws["G1"] = "Call Range Breakpoints (Tot Call)"
    ws["G1"].font = title_font
    ws["G2"] = "A max (Very slow, starts at 4)"; ws["H2"] = a_max
    ws["G3"] = "B max (Slow moving)"; ws["H3"] = b_max
    ws["G4"] = "C max (Medium moving)"; ws["H4"] = c_max
    ws["G5"] = "D = above C max (Fast moving)"
    ws["G6"] = "LA = calls 2-3, C1 = calls 1, C0 = calls 0 (fixed categories)"
    for r in (2, 3, 4):
        ws.cell(row=r, column=8).font = blue
    for r in (2, 3, 4, 5, 6):
        ws.cell(row=r, column=7).font = Font(name=FONT, italic=(r >= 5))
    ws.column_dimensions["G"].width = 34
    ws.column_dimensions["H"].width = 10

    ws["J1"] = "DN Price Range Breakpoints"
    ws["J1"].font = title_font
    ws["J2"] = "Limit of range 1"; ws["K2"] = p1
    ws["J3"] = "Limit of range 2"; ws["K3"] = p2
    ws["J4"] = "Limit of range 3"; ws["K4"] = p3
    ws["J5"] = "Limit of range 4"; ws["K5"] = p4
    ws["J6"] = "Above Limit 4 = high value"
    price_number_format = "0" if whole_number_price else "0.00"
    for r in (2, 3, 4, 5):
        ws.cell(row=r, column=11).font = blue
        ws.cell(row=r, column=11).number_format = price_number_format
    for r in (2, 3, 4, 5, 6):
        ws.cell(row=r, column=10).font = Font(name=FONT, italic=(r == 6))
    ws.column_dimensions["J"].width = 22
    ws.column_dimensions["K"].width = 10

    n_excluded = int((df["Price"] <= 0).sum())
    if n_excluded:
        ws["G7"] = (f"Note: {n_excluded} line item(s) have DN Price <= 0 and are excluded "
                    f"from the Price Range classification and both pivot tables below.")
        ws["G7"].font = Font(name=FONT, italic=True, size=9, color="808080")

    data_rng_D = f"$D$2:$D${last_row}"
    data_rng_E = f"$E$2:$E${last_row}"
    data_rng_B = f"$B$2:$B${last_row}"

    def build_pivot(start_row, title, value_formula_maker):
        ws.cell(row=start_row, column=7, value=title).font = title_font
        hdr = start_row + 1
        ws.cell(row=hdr, column=7, value="Price Range \\ Call Range").font = bold
        ws.cell(row=hdr, column=7).fill = header_fill
        for j, cat in enumerate(CALL_CATS, start=8):
            c = ws.cell(row=hdr, column=j, value=cat)
            c.font = bold; c.fill = header_fill; c.alignment = Alignment(horizontal="center")
        gt_col = 8 + len(CALL_CATS)
        c = ws.cell(row=hdr, column=gt_col, value="Grand Total")
        c.font = bold; c.fill = header_fill

        price_labels = [f'="to "&$K$2', f'="to "&$K$3', f'="to "&$K$4', f'="to "&$K$5', "high value"]
        first_data_row = hdr + 1
        for i, lbl in enumerate(price_labels):
            r = first_data_row + i
            ws.cell(row=r, column=7, value=lbl).font = black
            for j, cat in enumerate(CALL_CATS, start=8):
                col_letter = get_column_letter(j)
                formula = value_formula_maker(col_letter, f"$G{r}", hdr)
                ws.cell(row=r, column=j, value=formula).font = black
                ws.cell(row=r, column=j).number_format = "#,##0"
            first_cat_col = get_column_letter(8)
            last_cat_col = get_column_letter(7 + len(CALL_CATS))
            ws.cell(row=r, column=gt_col, value=f"=SUM({first_cat_col}{r}:{last_cat_col}{r})").font = bold
            ws.cell(row=r, column=gt_col).number_format = "#,##0"

        gt_row = first_data_row + len(price_labels)
        ws.cell(row=gt_row, column=7, value="Grand Total").font = bold
        last_data_row = first_data_row + len(price_labels) - 1
        for j in range(8, gt_col + 1):
            col_letter = get_column_letter(j)
            ws.cell(row=gt_row, column=j,
                    value=f"=SUM({col_letter}{first_data_row}:{col_letter}{last_data_row})").font = bold
            ws.cell(row=gt_row, column=j).number_format = "#,##0"
        for r in range(hdr, gt_row + 1):
            for j in range(7, gt_col + 1):
                ws.cell(row=r, column=j).border = border
        return gt_row

    def make_count_formula(col_letter, price_ref, hdr_row):
        return f"=COUNTIFS({data_rng_D},{col_letter}${hdr_row},{data_rng_E},{price_ref})"

    def make_sum_formula(col_letter, price_ref, hdr_row):
        return f"=SUMIFS({data_rng_B},{data_rng_D},{col_letter}${hdr_row},{data_rng_E},{price_ref})"

    end1 = build_pivot(9, "PIVOT 1 : Count of P/N", make_count_formula)
    start2 = end1 + 3
    build_pivot(start2, "PIVOT 2 : Total Calls", make_sum_formula)

    for col in ["H", "I", "J", "K", "L", "M", "N", "O"]:
        ws.column_dimensions[col].width = 12

    ws.freeze_panes = "A2"
    wb.save(output_path)
    print(f"Workbook written to {output_path}")

## Configuration

Edit the values below to match your input file, then run this cell and the **Execute** cell that follows.

- `INPUT_FILE`: path to your input .xlsx
- `OUTPUT_FILE`: path to write the optimized workbook
- `SHEET`: sheet name, or `None` to use the first sheet
- `PN_COL`, `PRICE_COL`, `CALLS_COL`: explicit column names, or `None` to auto-detect (partial, case-insensitive match on "P/N", "Price", "Call")
- `PRICE_CAP`: hard upper limit for "Limit of range 1" only (it will never be set above this value). Limits 2-4 are searched freely with no cap.
- `WHOLE_NUMBER_PRICE`: set to `True` to force all 4 price breakpoints to whole numbers

In [ ]:
INPUT_FILE = "call.xlsx"
OUTPUT_FILE = "OUTPUT.xlsx"
SHEET = "Sheet2"        # or None for first sheet
PN_COL = "P/N"          # or None to auto-detect
PRICE_COL = "DN Price"  # or None to auto-detect
CALLS_COL = "Sum of Tot Call"  # or None to auto-detect
PRICE_CAP = 10              # hard cap for Limit of range 1 only
WHOLE_NUMBER_PRICE = True   # set False to allow decimal price breakpoints

## Diagnostics (optional)

Run this if `find_price_breakpoints` raises a `ValueError`. It loads your data and shows:
- The overall price/calls distribution
- How many items and total calls fall at-or-below vs above `PRICE_CAP`
- Whether a valid 5-band split exists at all if the cap is ignored (i.e. is it your data, or specifically the cap, that's the problem)

In [ ]:
df_diag = load_data(INPUT_FILE, SHEET, PN_COL, PRICE_COL, CALLS_COL)
pdata_diag = df_diag[df_diag["Price"] > 0]

print(f"Rows with Price > 0: {len(pdata_diag)} (of {len(df_diag)} total)")
print(f"Price range: {pdata_diag['Price'].min()} to {pdata_diag['Price'].max()}")
print(f"Unique price values: {pdata_diag['Price'].nunique()}")
print()

below_cap = pdata_diag[pdata_diag["Price"] <= PRICE_CAP]
above_cap = pdata_diag[pdata_diag["Price"] > PRICE_CAP]
print(f"At/below PRICE_CAP ({PRICE_CAP}): {len(below_cap)} items, {below_cap['Calls'].sum()} total calls")
print(f"Above PRICE_CAP ({PRICE_CAP}):    {len(above_cap)} items, {above_cap['Calls'].sum()} total calls")
print()

print("Value counts near the cap (+/- a few values):")
near_cap = pdata_diag[(pdata_diag["Price"] >= PRICE_CAP - 5) & (pdata_diag["Price"] <= PRICE_CAP + 5)]
print(near_cap["Price"].value_counts().sort_index())
print()

# Does a valid split exist at all if we ignore the cap?
try:
    test_bps = find_price_breakpoints(df_diag, price_cap=pdata_diag["Price"].max(), whole_number=WHOLE_NUMBER_PRICE)
    print(f"Uncapped search DOES find a valid split: {test_bps}")
    print("-> Your data supports a valid 5-band split, just not with Limit 1 <= "
          f"{PRICE_CAP}. Try raising PRICE_CAP.")
except ValueError as e:
    print(f"Uncapped search also fails: {e}")
    print("-> The issue isn't the cap - no valid monotonic 5-band split exists "
          "for this dataset at all. You may need fewer bands, or to pre-clean "
          "the data (e.g. remove outliers).")

## Execute

In [ ]:
df = load_data(INPUT_FILE, SHEET, PN_COL, PRICE_COL, CALLS_COL)
print(f"Loaded {len(df)} rows.")

call_bps = find_call_breakpoints(df)
print(f"Call Range breakpoints -> A max={call_bps[0]}, B max={call_bps[1]}, C max={call_bps[2]}")

price_bps = find_price_breakpoints(
    df,
    price_cap=PRICE_CAP,
    whole_number=WHOLE_NUMBER_PRICE,
    force_cap=True,  # if the full pattern can't be satisfied, force Limit 1 = PRICE_CAP anyway
)
print(f"Price Range breakpoints -> {price_bps[0]}, {price_bps[1]}, {price_bps[2]}, {price_bps[3]}")

build_workbook(df, call_bps, price_bps, OUTPUT_FILE, whole_number_price=WHOLE_NUMBER_PRICE)